Here's the Q&A version — I've kept answers interview-ready (concise but substantive, the kind of thing you'd actually say out loud).

## 1. ADK Fundamentals

**Q: What is ADK and how does it differ from LangChain/CrewAI/Semantic Kernel?**
A: ADK is Google's open-source, code-first framework (released April 2025) for building, evaluating, and deploying agents and multi-agent systems — it's the same toolkit Google uses internally. Compared to LangChain (which is more of a general LLM-orchestration toolkit) or CrewAI (role-based multi-agent focus), ADK's differentiator is that it's built production-first: native OpenTelemetry tracing, built-in evaluation harnesses, first-class MCP/tool integration, and a clean deploy-anywhere story (Agent Engine, Cloud Run, GKE, or self-hosted). It's also model-agnostic — you're not locked into Gemini.

**Q: Core abstractions — Agent, Tool, Session, Runner, Event?**
A: An **Agent** wraps a model, instructions, and a set of tools/sub-agents. A **Tool** is a callable function/API the agent can invoke, described via schema so the LLM knows when and how to call it. A **Session** holds conversation/task state across turns. The **Runner** executes the agent loop — sending input, letting the model decide to call tools or respond, executing tool calls, feeding results back. **Events** are the structured trace of everything that happened (messages, tool calls, tool results) — this is what powers observability and evaluation.

**Q: LLM agent vs workflow agent (Sequential, Parallel, Loop)?**
A: An LLM agent lets the model itself decide the next step (dynamic reasoning). Workflow agents are deterministic orchestration primitives — Sequential runs sub-agents in a fixed order, Parallel fans out and merges results, Loop repeats until a condition is met. In support automation you typically mix both: a workflow agent enforces "always classify, then retrieve context, then diagnose" while the diagnosis step itself is an LLM agent reasoning freely within that stage.

## 2. Multi-Agent Orchestration

**Q: How would you decompose a support workflow into agents?**
A: Triage agent (classify severity/category) → retrieval agent (pull relevant runbooks/past tickets via RAG) → diagnostic agent (correlate logs/metrics, propose root cause) → decision agent (auto-remediate vs escalate) → action agent (executes approved fix) → feedback agent (writes outcome back to the knowledge base). Each agent has a narrow, testable responsibility — that's what makes the system debuggable and evaluable in production.

**Q: Sub-agents vs Agent-as-a-Tool — when do you use each?**
A: Sub-agents are for when you want the parent to hand off control and get a full conversational turn back (e.g., "diagnostic agent, take over and investigate"). Agent-as-a-Tool is for a narrower, single-shot capability call (e.g., "summarize this log file") where you don't want the sub-agent driving the conversation, just returning a result. Use sub-agents for stages of a pipeline, agent-as-tool for reusable capabilities.

**Q: How do you prevent infinite delegation loops?**
A: Hard step/turn limits per session, explicit termination conditions on Loop agents, and a supervisor/orchestrator agent that tracks how many times a ticket has bounced between agents — after N reassignments, force escalation to a human rather than letting agents keep handing off.

## 3. Tools & Enterprise Integration

**Q: How do you wire ADK into ServiceNow/Jira/Datadog?**
A: Expose them as tools — either direct API wrappers (functions with typed parameters) or, more cleanly, as MCP servers so any agent in the org can reuse the same ServiceNow/Datadog connector without reimplementing auth and pagination logic per team. The key production concern is defining a clean error contract in the tool's return schema so the LLM can distinguish "no data found" from "tool call failed."

**Q: How do you handle tool authentication securely?**
A: Never let the LLM see or handle credentials. The tool wrapper itself holds a service account or OAuth token (via Secret Manager/Workload Identity), scoped to least privilege for that specific system. The agent only sees "call get_incident_logs(service_id)" — the auth happens underneath, invisibly.

## 4. State & Memory

**Q: Short-term vs long-term memory in a support context?**
A: Session state holds the current ticket's investigation context (logs pulled, hypotheses tried). Long-term memory (usually backed by a vector store or a structured DB) holds cross-ticket knowledge — "this same error on this service was resolved by X three weeks ago." You write successful resolutions back into long-term memory so the system gets smarter over time.

**Q: How do you avoid context overflow in a long investigation?**
A: Summarize and compact older tool outputs instead of keeping raw logs in context; keep only the running hypothesis and key evidence in session state; push full raw data to an external store and give the agent a "fetch details" tool instead of inlining everything.

## 5. Production, Deployment & Scaling

**Q: Agent Engine vs Cloud Run vs GKE — trade-offs?**
A: Agent Engine is the serverless, purpose-built runtime for Python agent workflows — least ops overhead, good default for most agent backends. Cloud Run fits when you need a custom API/UI layer around the agent or want standard container-based serverless scaling. GKE is for when you need full control — custom networking, sidecars, strict compliance isolation, or you're already standardized on Kubernetes org-wide. For an enterprise support agent handling sensitive incident data, GKE or Agent Engine with VPC-SC are common choices.

**Q: How do you scale for ticket volume spikes?**
A: Stateless agent workers behind a queue (Pub/Sub) so ticket intake and agent processing are decoupled; autoscaling on the runtime; and rate-limiting/backpressure on the LLM calls themselves so a spike doesn't blow your model quota or budget.

## 6. Observability & Evaluation

**Q: How do you debug a misbehaving agent in production?**
A: ADK's built-in OpenTelemetry tracing gives you the full event trace — every model call, every tool call and its input/output, every decision branch. You'd pull the trace for the failing session, look at where the tool call returned unexpected data or where the model's reasoning diverged, and reproduce it in an eval harness.

**Q: How do you evaluate agent quality before shipping?**
A: Build a golden dataset of real (anonymized) past tickets with known-correct outcomes, run the agent against them automatically on every change, and score on resolution accuracy, correct escalation decisions, and tool-call correctness — not just final-answer similarity. Treat it like regression testing for a non-deterministic system: track pass-rate trends, not single runs.

## 7. Reliability, Guardrails, Safety

**Q: How do you stop the agent from taking destructive actions unsupervised?**
A: Tier your tools — read-only tools (fetch logs, query metrics) execute freely; write/action tools (restart a service, close a ticket) require an explicit human-approval gate before execution. This is usually modeled as the agent proposing an action and a human-in-the-loop step confirming it, rather than the agent having unmediated write access.

**Q: How do you handle prompt injection from ticket/log content?**
A: Treat all ingested ticket text and log content as untrusted data, not instructions — the system prompt should explicitly state that content retrieved via tools is data to analyze, never commands to follow. Sanitize/strip anything that looks like an instruction override before it reaches the model, and keep the action-taking tools behind the approval gate regardless of what the "ticket" says.

## 8. Scenario Design Question (very likely to be asked)
Be ready to whiteboard: ticket intake → triage agent classifies severity/category → retrieval agent pulls relevant runbooks/similar past incidents (RAG) → diagnostic agent correlates with live logs/metrics via tools → decision agent proposes auto-fix or escalation → human approval gate for any write action → execution agent performs the approved fix → outcome logged back into the knowledge base for future retrieval. Mention audit logging of every agent decision (important for regulated industries) and idempotent actions (so a retried fix doesn't double-apply).

Want this as a printable PDF or Word doc to glance at before the interview?